# 🎯 CosyVoice 300M-SFT — Türkisch Test
**Alles in einer Zelle. T4 GPU. Einmal ausführen.**

In [ ]:
#@title ⚙️ Schritt 1: Alles installieren + Modell laden (~15 Min)
import subprocess, sys, os

# CosyVoice klonen
if not os.path.exists('/content/CosyVoice'):
    print('CosyVoice klonen...')
    os.system('git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git')

sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

# Alle Pakete installieren
print('Pakete installieren...')
os.system('pip install -q -r /content/CosyVoice/requirements.txt')
os.system('pip install -q pyworld hydra-core lightning openai-whisper omegaconf')
print('Pakete OK!')

# PyTorch 2.6 fix
import torch
_orig = torch.load
torch.load = lambda *a, **kw: _orig(*a, **{**kw, 'weights_only': False})

# Modell laden
print('Modell laden (CosyVoice-300M-SFT)...')
from cosyvoice.cli.cosyvoice import CosyVoice
model = CosyVoice('iic/CosyVoice-300M-SFT')
print('✅ FERTIG! Jetzt Schritt 2 ausführen.')


In [ ]:
#@title 🎤 Schritt 2: Stimme hochladen + Türkisch generieren
from google.colab import files
import torchaudio, whisper, sys, torch
from IPython.display import Audio, display
sys.path.insert(0, '/content/CosyVoice')

# Stimme hochladen
print('Stimme hochladen (WAV/MP3):')
uploaded = files.upload()
voice_file = list(uploaded.keys())[0]

# Stimme laden - Mono - 8 Sekunden
speech, sr = torchaudio.load(voice_file)
if sr != 16000:
    speech = torchaudio.functional.resample(speech, sr, 16000)
if speech.shape[0] == 2:
    speech = speech.mean(dim=0, keepdim=True)
speech_8s = speech[:, :16000*8]
torchaudio.save('/content/stimme_8s.wav', speech_8s, 16000)
print(f'✅ Stimme: {speech_8s.shape[1]/16000:.1f} Sek, Mono, 16kHz')

# Transkription der 8 Sekunden
print('Transkribiere...')
w = whisper.load_model('tiny')
result = w.transcribe('/content/stimme_8s.wav', language='tr')
prompt_text = result['text'].strip()
print(f'Prompt Text: {prompt_text}')

# Generieren
ziel_text = 'Merhaba, benim adım Ahrar. Ben bir Türk yazarıyım. Kitaplarımda Osmanlı tarihini anlatıyorum. Sûfî düşünce benim ana temam.' #@param {type:"string"}
print(f'Generiere Türkisch...')
for i, res in enumerate(model.inference_zero_shot(
    ziel_text, prompt_text, speech_8s, stream=False
)):
    torchaudio.save(f'/content/cosy_output.wav', res['tts_speech'], model.sample_rate)
    print('✅ Fertig! Anhören:')
    display(Audio('/content/cosy_output.wav'))
